# TideTrace, SIH26143, U-Net training on Kaggle

Kaggle is the factory. The laptop is the product.

This notebook trains the 3-class segmenter and exports one checkpoint under
80 MB. It never runs at demo time, and the judged demo never depends on it:
if the checkpoint is absent the app falls back to the published dB baseline
and still runs clauses (a), (b) and (c).

**Before you start**, prepare tiles locally and upload the folder as a Kaggle
dataset:

```
python scripts/download_zenodo_subset.py --file 01_Train_Val_Oil_Spill_mask.7z --extract
python scripts/prepare_tiles.py --root data/zenodo --out data/tiles \
    --oil 400 --lookalike 250 --empty 150
```

Settings: Accelerator GPU P100, Internet on for the pip install only.


## 1. Environment


In [ ]:
!pip -q install segmentation-models-pytorch==0.3.4 albumentations==1.4.14
import torch, os, sys, json
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


## 2. Get the TideTrace code

Upload the repository as a Kaggle dataset, or clone it. Only `app/ml`,
`app/geo` and `app/config.py` are needed here.


In [ ]:
REPO = '/kaggle/input/tidetrace-repo'   # adjust to your dataset name
TILES = '/kaggle/input/tidetrace-tiles'  # adjust to your tiles dataset

import shutil, pathlib
work = pathlib.Path('/kaggle/working/tidetrace')
if not work.exists():
    shutil.copytree(REPO, work)
sys.path.insert(0, str(work))
os.environ['TIDETRACE_DATA'] = '/kaggle/working/data'
os.environ['TIDETRACE_MODELS'] = '/kaggle/working/models'

from app import config
from app.ml import train as train_mod
print('tile size', config.TILE, 'overlap', config.TILE_OVERLAP)


## 3. Inspect the tiles

Check the class balance before spending GPU hours on it. If the oil class is
a rounding error, fix the sampler, not the learning rate.


In [ ]:
import numpy as np, glob
idx = json.load(open(os.path.join(TILES, 'index.json')))
print('tiles', idx['tiles'], 'per class', idx['counts'])
print('normalisation', idx['normalisation'])

files = sorted(glob.glob(os.path.join(TILES, '*.npz')))[:200]
px = np.zeros(3, dtype=np.int64)
for f in files:
    lab = np.load(f)['label']
    px += np.bincount(lab.ravel(), minlength=3)
print('pixel share sea / look-alike / oil:', np.round(px / px.sum(), 4))


## 4. Look at a few tiles

Always eyeball the data. A silent labelling bug is cheaper to find here than
after a twelve hour run.


In [ ]:
import matplotlib.pyplot as plt

picks = [f for f in files if np.load(f)['label'].max() == 2][:4]
fig, axes = plt.subplots(2, len(picks), figsize=(4 * len(picks), 8))
for j, f in enumerate(picks):
    z = np.load(f)
    axes[0, j].imshow(z['image'][0], cmap='gray'); axes[0, j].set_title('VV dB')
    axes[1, j].imshow(z['label'], vmin=0, vmax=2, cmap='viridis')
    axes[1, j].set_title('0 sea / 1 look-alike / 2 oil')
    for a in (axes[0, j], axes[1, j]): a.axis('off')
plt.tight_layout(); plt.show()


## 5. Sanity run

Five epochs on a small slice. This proves the loaders, the loss and the IoU
computation. Roughly 25 to 45 minutes on a T4 or P100. Do this first, always.


In [ ]:
report = train_mod.train(
    tile_dir=TILES,
    out_path='/kaggle/working/models/sanity.pt',
    epochs=5,
    batch_size=8,
    max_train_tiles=400,
    workers=2,
)
print(json.dumps(report['best'], indent=2))


## 6. Full run

30 to 40 epochs on the whole subset. Batch 8 on P100, 4 on T4. Expect 6 to 12
hours on a P100 for a 300 to 400 image subset. Save the best `iou_oil`.


In [ ]:
report = train_mod.train(
    tile_dir=TILES,
    out_path='/kaggle/working/models/oil_unet_best.pt',
    arch='UnetPlusPlus',
    encoder='timm-efficientnet-b0',
    epochs=40,
    batch_size=8,
    lr=1e-4,
    workers=2,
)


## 7. The metrics card

This is the table that shows you are not a wrapper: the trained model next to
the published dB threshold baseline, measured on the same validation tiles.


In [ ]:
best, base = report['best'], report['baseline']
rows = [
    ('IoU oil',        base['iou_oil'],        best['iou_oil']),
    ('IoU look-alike', base['iou_lookalike'],  best['iou_lookalike']),
    ('IoU sea',        base['iou_sea'],        best['iou_sea']),
    ('pixel accuracy', base['pixel_accuracy'], best['pixel_accuracy']),
]
print('%-16s %12s %12s %10s' % ('metric', '-22 dB base', 'U-Net', 'delta'))
for name, b, m in rows:
    print('%-16s %12.4f %12.4f %+10.4f' % (name, b, m, m - b))


## 8. Export

Copy `oil_unet_best.pt` into `models/` on the demo laptop. Nothing else
changes. The app picks it up on next start and the top bar switches from
`dB BASELINE` to `U-NET LOADED`.

Do **not** serve the judged demo from Kaggle.


In [ ]:
import os
p = '/kaggle/working/models/oil_unet_best.pt'
print(p, '%.1f MB' % (os.path.getsize(p) / 1e6))
assert os.path.getsize(p) < 80e6, 'checkpoint must stay under 80 MB'
